## AI Multi-Platform Search and Summarization Program

This program aims to provide important answers from various online sources (Google, YouTube, and conceptually TikTok) and offer options for deeper exploration.

**Key Components:**
1.  **API Key Setup**: You will need API keys for Google services (like YouTube Data API, potentially Google Custom Search API) and possibly the Gemini API for summarization.
2.  **Search Functions**: Functions to query each platform.
3.  **Information Extraction & Summarization**: Processing search results to extract key information and summarizing it.
4.  **User Interface**: A simple loop to interact with the user and offer "more thinking" options.

**Important Considerations:**
*   **API Keys**: You'll need to obtain API keys for YouTube and Gemini. For Google Search, you might use a custom search API. TikTok access is significantly more challenging due to lack of a public search API and strict web scraping policies.
*   **Rate Limits and Terms of Service**: Always be mindful of API rate limits and the terms of service for each platform to ensure ethical and compliant usage.
*   **"Important Answer" and "More Thinking"**: These require sophisticated natural language processing (NLP) and potentially another LLM to refine answers or perform deeper analysis based on user prompts.

### 1. Set Up API Keys

Before running the search functions, you'll need to securely store and access your API keys. We'll use `google.colab.userdata` for this. For YouTube and Gemini, create keys in their respective developer consoles and add them to Colab secrets (the '🔑' icon in the left panel) under the names `YOUTUBE_API_KEY` and `GOOGLE_API_KEY`.

In [ ]:
# Install necessary libraries if you haven't already
# !pip install google-api-python-client google-generativeai

import os
from google.colab import userdata
import google.generativeai as genai

# Retrieve API keys from Colab secrets
YOUTUBE_API_KEY = userdata.get('YOUTUBE_API_KEY', '') # YouTube Data API Key
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY', '') # Gemini API Key

# Configure Gemini API
if GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)
    gemini_model = genai.GenerativeModel('gemini-1.5-flash') # Or gemini-1.0-pro
    print("Gemini API configured.")
else:
    print("Warning: GOOGLE_API_KEY not found. Gemini API will not be available.")

print("Please ensure YOUTUBE_API_KEY and GOOGLE_API_KEY are set in Colab secrets.")

### 2. Search Functions

#### Google Search (Conceptual)

Direct scraping of Google Search results is against their terms of service and prone to breakage. For programmatic access, you would typically use a Google Custom Search Engine API. This requires setup in the Google Cloud Console and an API key. For demonstration, we'll imagine a function that could *theoretically* fetch results.

In [ ]:
import requests

def google_search(query):
    print(f"\nSearching Google for: '{query}' (conceptual)")
    # In a real scenario, you'd use Google Custom Search API or similar.
    # Example with a placeholder:
    if not YOUTUBE_API_KEY: # Using YOUTUBE_API_KEY as a proxy for an actual Google API key for this example
        print("  (Skipping Google search due to missing API key for demonstration)")
        return [{"title": "Example Google Result 1", "link": "http://example.com/g1", "snippet": "A snippet of information from Google."},
                {"title": "Example Google Result 2", "link": "http://example.com/g2", "snippet": "Another snippet about the topic."}]

    # Placeholder for actual Google Custom Search API call
    # Example: url = f"https://www.googleapis.com/customsearch/v1?key={YOUR_GOOGLE_SEARCH_API_KEY}&cx={YOUR_CSE_ID}&q={query}"
    # response = requests.get(url).json()
    # Process 'response' to extract titles, links, snippets

    print("  (Actual Google Search would require Google Custom Search API key and CSE ID)")
    return [{"title": "Real Google Result 1", "link": "http://realgoogle.com/r1", "snippet": "An actual snippet from a Google search."},
            {"title": "Real Google Result 2", "link": "http://realgoogle.com/r2", "snippet": "More information on your query."}]

#### YouTube Search

We can use the `google-api-python-client` to interact with the YouTube Data API. This requires a `YOUTUBE_API_KEY`.

In [ ]:
from googleapiclient.discovery import build

def youtube_search(query, max_results=3):
    print(f"\nSearching YouTube for: '{query}'")
    if not YOUTUBE_API_KEY:
        print("  Error: YOUTUBE_API_KEY not found in Colab secrets. Skipping YouTube search.")
        return []

    try:
        youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)
        request = youtube.search().list(
            q=query,
            part='snippet',
            type='video',
            maxResults=max_results
        )
        response = request.execute()

        results = []
        for item in response.get('items', []):
            video_id = item['id']['videoId']
            title = item['snippet']['title']
            description = item['snippet']['description']
            link = f"https://www.youtube.com/watch?v={video_id}"
            results.append({"title": title, "link": link, "description": description})
        return results
    except Exception as e:
        print(f"  Error during YouTube search: {e}")
        print("  Please check your YOUTUBE_API_KEY and API quotas.")
        return []

#### TikTok Search (Limitations)

Programmatic access to TikTok's search functionality is extremely limited without an official API for general search. Web scraping TikTok is challenging due to dynamic content, CAPTCHAs, and is often against their terms of service. For a real application, you would need to explore third-party services or very advanced, ethically compliant (and potentially brittle) scraping techniques, which are beyond the scope of this notebook. We'll include a placeholder function to acknowledge this.

In [ ]:
def tiktok_search(query):
    print(f"\nSearching TikTok for: '{query}' (limited/conceptual)")
    print("  Note: Direct programmatic access to TikTok search is very difficult without an official API.")
    print("  Returning a conceptual result.")
    return [{"title": "TikTok Concept Video", "link": "https://tiktok.com/tag/{query}", "snippet": "Short video content related to your query on TikTok."}]

### 3. Information Extraction and Summarization

To get the "important answer" and offer "more thinking", we can leverage a Large Language Model (LLM) like Gemini. We'll pass the combined search results to Gemini for summarization.

In [ ]:
def summarize_with_gemini(query, search_results):
    if not GOOGLE_API_KEY:
        return "Cannot summarize: GOOGLE_API_KEY not configured."

    combined_text = f"User asked: '{query}'\n\nHere are some search results:\n\n"
    for platform, results_list in search_results.items():
        combined_text += f"--- From {platform.upper()} ---\n"
        if results_list:
            for i, result in enumerate(results_list):
                combined_text += f"{i+1}. Title: {result.get('title', 'N/A')}\n   Link: {result.get('link', 'N/A')}\n   Snippet/Description: {result.get('snippet', result.get('description', 'N/A'))[:200]}...\n\n"
        else:
            combined_text += "No results found.\n\n"

    prompt = f"Based on the following search results for the query '{query}', provide a concise and important answer. Then, suggest 1-2 follow-up questions for 'more thinking'.\n\n{combined_text}\n\nImportant Answer:\n"

    try:
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error summarizing with Gemini: {e}"

### 4. Main AI Program

This is the main loop where you can ask questions and choose the "more thinking" option.

In [ ]:
def run_ai_program():
    print("\n--- AI Multi-Platform Search and Summarization ---")
    print("Type 'exit' to quit.")

    while True:
        user_query = input("\nWhat would you like to know? ")
        if user_query.lower() == 'exit':
            break

        print("Searching...")

        all_results = {}

        # Perform searches
        all_results['google'] = google_search(user_query)
        all_results['youtube'] = youtube_search(user_query)
        all_results['tiktok'] = tiktok_search(user_query)

        # Summarize results
        print("\nAnalyzing and summarizing important answers...")
        summary_and_thinking = summarize_with_gemini(user_query, all_results)
        print(summary_and_thinking)

        # Offer 'more thinking' option
        more_thinking_choice = input("\nWould you like more thinking/follow-up on this topic? (yes/no): ")
        if more_thinking_choice.lower() == 'yes':
            print("Please elaborate on what aspects you'd like to explore further, or ask a follow-up question:")
            follow_up_query = input("Your follow-up: ")
            # This can be fed back into the summarization or even search functions
            print("\nPerforming deeper analysis/search for your follow-up... (conceptual)")
            # For a real system, you'd rerun summarize_with_gemini with the new context
            # or even trigger specific new searches based on the follow_up_query.
            print(summarize_with_gemini(f"{user_query} - follow-up: {follow_up_query}", all_results)) # Simplified

    print("\nThank you for using the AI program!")

# To run the program:
run_ai_program()


## Movies and Series

Add your movie and series related code or notes here!

### My Movies and Series Collection

Let's create a simple database of your movies and series using a pandas DataFrame. You can expand this with more titles and details.

In [ ]:
import pandas as pd

# Create a DataFrame to store movie and series data
movies_series_data = {
    'Title': [
        'Stranger Things',
        'The Crown',
        'The Queen\'s Gambit',
        'Interstellar',
        'Pulp Fiction',
        'The Mandalorian',
        'Arcane'
    ],
    'Type': [
        'Series',
        'Series',
        'Series',
        'Movie',
        'Movie',
        'Series',
        'Series'
    ],
    'Genre': [
        'Sci-Fi, Horror',
        'Drama, Historical',
        'Drama',
        'Sci-Fi, Adventure',
        'Crime, Drama',
        'Sci-Fi, Action',
        'Animation, Action'
    ],
    'Year': [
        2016,
        2016,
        2020,
        2014,
        1994,
        2019,
        2021
    ],
    'Rating': [
        8.7,
        8.7,
        8.6,
        8.6,
        8.9,
        8.7,
        9.0
    ]
}

my_collection_df = pd.DataFrame(movies_series_data)

print("Here's your current movie and series collection:")
display(my_collection_df)

Now that you have a basic collection, what 'Netflix-like' features would you like to implement?

For example:
*   **Search**: Allow users to search for titles.
*   **Filter**: Filter by type (Movie/Series), genre, or year.
*   **Add new content**: A function to easily add new movies or series.
*   **Recommendations**: A simple system to suggest similar titles.